# 📘 CodeBook Social Network Analysis
## Notebook 02 — Data Cleaning

**Author:** Aditi Chaudhary  
**Tech Stack:** Pure Python · JSON

---

## 🎯 Why Data Cleaning Matters

> **"Garbage in, garbage out."**

If we run our recommendation algorithms on dirty data,  
we will get wrong or misleading results.

In this notebook we fix **4 specific data quality problems**:

| # | Problem | Impact if not fixed |
|---|---|---|
| 1 | Users with empty names | Unnamed users break display and recommendation output |
| 2 | Duplicate friend IDs | Mutual friend count gets inflated — wrong recommendations |
| 3 | Inactive users | Stale accounts pollute the network graph |
| 4 | Duplicate pages | Same page appears twice — recommendation engine double-counts it |

---

## 📂 In This Notebook

1. Load raw data
2. Run each cleaning step with explanation
3. Compare before vs after
4. Save cleaned data to `data/cleaned_data.json`

In [ ]:
import sys, os, json

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data_loader  import load_data
from src.data_cleaning import (
    remove_empty_names,
    remove_duplicate_friends,
    remove_inactive_users,
    remove_duplicate_pages,
    clean_data,
    save_cleaned_data
)

import copy

data_path = os.path.join(project_root, 'data', 'raw_data.json')
raw_data  = load_data(data_path)
print(f'Raw users: {len(raw_data["users"])} | Raw pages: {len(raw_data["pages"])}')

---
## 🧹 Step 1 — Remove Users With Empty Names

**Problem:** Some user accounts have a blank `name` field.  
These are incomplete registrations — they can't appear in recommendations  
or friend suggestions, so we remove them.

**Rule:** If `name.strip()` is empty → remove the user.

In [ ]:
users_copy = copy.deepcopy(raw_data['users'])

print('Before cleaning — all users:')
for u in users_copy:
    name = u.get('name', '').strip()
    flag = '⚠️  EMPTY' if not name else '✅'
    print(f'  [{u["id"]}] "{u["name"]}" {flag}')

print()
cleaned_users, removed = remove_empty_names(users_copy)
print(f'\nAfter: {len(cleaned_users)} users retained, {len(removed)} removed.')

---
## 🧹 Step 2 — Remove Duplicate Friends

**Problem:** Some friend lists contain the same user ID more than once.  
This happens due to bugs in the app (e.g., double-tap on 'Add Friend').

**Impact:** If Aarav's friend list has `[u003, u003]`, counting mutual friends  
with another user would incorrectly double-count u003.

**Rule:** Keep only the first occurrence of each friend ID.

In [ ]:
print('Checking for duplicate friends in raw data:')
for u in raw_data['users']:
    friends = u.get('friends', [])
    unique  = list(dict.fromkeys(friends))   # preserve order, deduplicate
    if len(friends) != len(unique):
        print(f'  ⚠️  [{u["id"]}] {u["name"]}: {friends} → has duplicates!')

print()
cleaned_users, total_dupes = remove_duplicate_friends(cleaned_users)

---
## 🧹 Step 3 — Remove Inactive Users

**Problem:** Some accounts haven't been used in over a year.  
Recommending these users to others is a bad experience — the person  
is unlikely to respond.

**Threshold:** Any user with `last_active_days_ago > 180` is removed.  
(That's 6 months — a reasonable inactivity cutoff.)

In [ ]:
INACTIVE_THRESHOLD = 180   # days

print(f'Inactivity threshold: {INACTIVE_THRESHOLD} days')
print('Checking activity for each user:')
for u in cleaned_users:
    days = u.get('last_active_days_ago', 0)
    flag = '⚠️  INACTIVE' if days > INACTIVE_THRESHOLD else '✅ Active'
    print(f'  [{u["id"]}] {u["name"]:<20} | Last active: {days:>4} days ago | {flag}')

print()
cleaned_users, inactive = remove_inactive_users(cleaned_users, INACTIVE_THRESHOLD)

---
## 🧹 Step 4 — Remove Duplicate Pages

**Problem:** The `pages` list contains the same page ID more than once.  
This happened because the same page was added twice to the database.

**Impact:** A duplicate page would appear twice in recommendations,  
making the app look buggy.

**Rule:** Keep only the first occurrence of each `page id`.

In [ ]:
pages_copy = copy.deepcopy(raw_data['pages'])

print('All pages (raw — before deduplication):')
seen = set()
for p in pages_copy:
    flag = '⚠️  DUPLICATE' if p['id'] in seen else '✅'
    print(f'  [{p["id"]}] {p["name"]} {flag}')
    seen.add(p['id'])

print()
cleaned_pages, dupes_removed = remove_duplicate_pages(pages_copy)

---
## ✅ Step 5 — Run Full Pipeline & Compare Before vs After

In [ ]:
# Run the complete cleaning pipeline
cleaned_data = clean_data(raw_data, inactive_threshold=180)

# Before vs After comparison
print('\n📊 Before vs After Cleaning:')
print(f'  {"Metric":<25} {"Before":>8} {"After":>8}')
print(f'  {"-"*43}')
print(f'  {"Total Users":<25} {len(raw_data["users"]):>8} {len(cleaned_data["users"]):>8}')
print(f'  {"Total Pages":<25} {len(raw_data["pages"]):>8} {len(cleaned_data["pages"]):>8}')

raw_friend_count     = sum(len(u.get('friends', [])) for u in raw_data['users'])
cleaned_friend_count = sum(len(u.get('friends', [])) for u in cleaned_data['users'])
print(f'  {"Total Friend Entries":<25} {raw_friend_count:>8} {cleaned_friend_count:>8}')

---
## 💾 Step 6 — Save Cleaned Data

In [ ]:
cleaned_path = os.path.join(project_root, 'data', 'cleaned_data.json')
save_cleaned_data(cleaned_data, cleaned_path)

# Verify by reloading
verify = load_data(cleaned_path)
print(f'\n✅ Verification: {len(verify["users"])} users, {len(verify["pages"])} pages in cleaned file.')

---
## ✅ Notebook 02 — Summary

| Cleaning Step | Issue Fixed | Method |
|---|---|---|
| Empty name removal | Users with no name | `str.strip()` check |
| Duplicate friends | Inflated friend counts | `set` deduplication |
| Inactive users | Stale accounts | Threshold comparison |
| Duplicate pages | Double-listed pages | ID-based dedup |

**Next → Notebook 03: People You May Know**